# SQL Analysis — Olist E-commerce
Using SQLite to query Olist data with pure SQL.
Same questions as Python — different tool, same answers.
This file contains 5 business questions answered in SQL.

In [1]:
import sqlite3
import pandas as pd

# Path to your datasets folder
path = r"E:\NILE TECH\DataAnalyst90Days\Phase1_Olist\datasets\\"

# Create a new SQLite database file
conn = sqlite3.connect(r"E:\NILE TECH\DataAnalyst90Days\Phase1_Olist\olist.db")

# Load each CSV into the database as a table
tables = {
    'orders'        : 'olist_orders_dataset.csv',
    'order_items'   : 'olist_order_items_dataset.csv',
    'order_payments': 'olist_order_payments_dataset.csv',
    'order_reviews' : 'olist_order_reviews_dataset.csv',
    'customers'     : 'olist_customers_dataset.csv',
    'sellers'       : 'olist_sellers_dataset.csv',
    'products'      : 'olist_products_dataset.csv',
    'translations'  : 'product_category_name_translation.csv'
}

for table_name, filename in tables.items():
    df = pd.read_csv(path + filename)
    df.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"✓ {table_name} loaded — {len(df):,} rows")

print("\nDatabase ready! All tables loaded.")

✓ orders loaded — 99,441 rows
✓ order_items loaded — 112,650 rows
✓ order_payments loaded — 103,886 rows
✓ order_reviews loaded — 99,224 rows
✓ customers loaded — 99,441 rows
✓ sellers loaded — 3,095 rows
✓ products loaded — 32,951 rows
✓ translations loaded — 71 rows

Database ready! All tables loaded.


In [2]:
def run_sql(query):
    """Run a SQL query and return results as a pandas DataFrame"""
    return pd.read_sql_query(query, conn)

print("Helper function ready. Use run_sql('YOUR QUERY HERE') to run SQL.")

Helper function ready. Use run_sql('YOUR QUERY HERE') to run SQL.


Query 1 — how many orders exist and what statuses?

In [4]:
## Count orders by status — your first GROUP BY
run_sql("""
SELECT
    order_status,
    COUNT(*) AS order_count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM orders), 1) AS percentage
FROM orders
GROUP BY order_status
ORDER BY order_count DESC
""")

,order_status,order_count,percentage
0,delivered,96478,97.0
1,shipped,1107,1.1
2,canceled,625,0.6
3,unavailable,609,0.6
4,invoiced,314,0.3
5,processing,301,0.3
6,created,5,0.0
7,approved,2,0.0


Query 2 — top 10 product categories by order volume

In [5]:
## Same question you answered in Python with groupby()
## Now answering it in SQL with JOIN + GROUP BY
run_sql("""
SELECT
    t.product_category_name_english AS category,
    COUNT(oi.order_id) AS order_count,
    ROUND(SUM(oi.price), 2) AS total_revenue,
    ROUND(AVG(oi.price), 2) AS avg_price
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
JOIN translations t ON p.product_category_name = t.product_category_name
GROUP BY category
ORDER BY order_count DESC
LIMIT 10
""")

,category,order_count,total_revenue,avg_price
0,bed_bath_table,11115,1036988.68,93.30
1,health_beauty,9670,1258681.34,130.16
2,sports_leisure,8641,988048.97,114.34
3,furniture_decor,8334,729762.49,87.56
4,computers_accessories,7827,911954.32,116.51
5,housewares,6964,632248.66,90.79
6,watches_gifts,5991,1205005.68,201.14
7,telephony,4545,323667.53,71.21
8,garden_tools,4347,485256.46,111.63
9,auto,4235,592720.11,139.96


Query 3 — which sellers have the most orders?

In [6]:
run_sql("""
SELECT
    oi.seller_id,
    s.seller_city,
    s.seller_state,
    COUNT(oi.order_id) AS total_orders,
    ROUND(SUM(oi.price), 2) AS total_revenue
FROM order_items oi
JOIN sellers s ON oi.seller_id = s.seller_id
GROUP BY oi.seller_id
ORDER BY total_orders DESC
LIMIT 10
""")

,seller_id,seller_city,seller_state,total_orders,total_revenue
0,6560211a19b47992c3666cc44a7e94c0,sao paulo,SP,2033,123304.83
1,4a3ca9315b744ce9f8e9374361493884,ibitinga,SP,1987,200472.92
2,1f50f920176fa81dab994f9023523100,sao jose do rio preto,SP,1931,106939.21
3,cc419e0650a3c5ba77189a1882b7556a,santo andre,SP,1775,104288.42
4,da8622b14eb17ae2831f4ac5b9dab84a,piracicaba,SP,1551,160236.57
5,955fee9216a65b617aa5c0531780ce60,sao paulo,SP,1499,135171.70
6,1025f0e2d44d7041d6cf58b6550e0bfa,sao paulo,SP,1428,138968.55
7,7c67e1448b00f6e969d365cea6b010ab,itaquaquecetuba,SP,1364,187923.89
8,ea8482cd71df3c1969d7b9473ff13abc,sao paulo,SP,1203,37177.52
9,7a67c85e85bb2ce8582c35f2203ad736,sao paulo,SP,1171,141745.53


Query 4 — payment method breakdown

In [7]:
run_sql("""
SELECT
    payment_type,
    COUNT(*) AS transaction_count,
    ROUND(AVG(payment_value), 2) AS avg_payment,
    ROUND(SUM(payment_value), 2) AS total_revenue
FROM order_payments
GROUP BY payment_type
ORDER BY transaction_count DESC
""")


,payment_type,transaction_count,avg_payment,total_revenue
0,credit_card,76795,163.32,12542084.19
1,boleto,19784,145.03,2869361.27
2,voucher,5775,65.70,379436.87
3,debit_card,1529,142.57,217989.79
4,not_defined,3,0.00,0.00


Query 5 — revenue by Brazilian state

In [8]:
run_sql("""
SELECT
    c.customer_state AS state,
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(SUM(oi.price), 2) AS total_revenue
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.order_status = 'delivered'
GROUP BY state
ORDER BY total_revenue DESC
LIMIT 10
""")

,state,total_orders,total_revenue
0,SP,40501,5067633.16
1,RJ,12350,1759651.13
2,MG,11354,1552481.83
3,RS,5345,728897.47
4,PR,4923,666063.51
5,SC,3546,507012.13
6,BA,3256,493584.14
7,DF,2080,296498.41
8,GO,1957,282836.70
9,ES,1995,268643.45
